# 10.15 — ControlNet & conditioning
ControlNet adds a trainable conditioning path that can steer a frozen diffusion model without destroying it. Latent diffusion supplies the base generator and residual conditioning supplies pose, edge, depth, or layout control. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

SEED = 1014
rng = np.random.default_rng(SEED)


def normalize01(x):
    arr = np.asarray(x, dtype=float)
    lo = float(np.min(arr))
    hi = float(np.max(arr))
    span = hi - lo
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / span


def synthetic_faces(n=64, size=16, seed=SEED):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    faces = []
    for i in range(n):
        face = 0.25 + 0.25 * np.exp(-2.0 * (xx ** 2 + yy ** 2))
        eye_y = -0.30 + local_rng.normal(0.0, 0.03)
        eye_dx = 0.35 + local_rng.normal(0.0, 0.02)
        mouth_y = 0.45 + local_rng.normal(0.0, 0.04)
        face -= 0.35 * np.exp(-90.0 * ((xx - eye_dx) ** 2 + (yy - eye_y) ** 2))
        face -= 0.35 * np.exp(-90.0 * ((xx + eye_dx) ** 2 + (yy - eye_y) ** 2))
        face += 0.14 * np.exp(-70.0 * (xx ** 2 + (yy - 0.06) ** 2))
        face -= 0.20 * np.exp(-55.0 * (xx ** 2 + (yy - mouth_y) ** 2))
        face += local_rng.normal(0.0, 0.025, size=(size, size))
        faces.append(normalize01(face))
    return np.asarray(faces).reshape(n, size * size)


def load_tiny_faces():
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(download_if_missing=False, shuffle=True, random_state=SEED)
        data = faces.data[:64]
        return data, (64, 64), "Olivetti faces cache"
    except Exception:
        return synthetic_faces(), (16, 16), "synthetic face fallback"


def make_f9_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    d1 = local_rng.normal(0.0, 1.0, size=(80, 1))
    d2, _ = make_moons(n_samples=120, noise=0.08, random_state=seed)
    d3, _ = make_blobs(n_samples=144, centers=4, n_features=6, cluster_std=0.65, random_state=seed)
    digits = load_digits()
    d4 = digits.data[:80] / 16.0
    faces, face_shape, face_source = load_tiny_faces()
    ladder = [
        {"name": "D1 gaussian", "data": d1, "shape": (1, 1), "kind": "vector"},
        {"name": "D2 two moons", "data": d2, "shape": (1, 2), "kind": "points"},
        {"name": "D3 mixture", "data": d3, "shape": (2, 3), "kind": "vector"},
        {"name": "D4 sklearn digits", "data": d4, "shape": (8, 8), "kind": "image"},
        {"name": "D5 faces (" + face_source + ")", "data": faces, "shape": face_shape, "kind": "image"},
    ]
    return ladder


def centered_scaled(data):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(np.asarray(data, dtype=float))
    return scaled, scaler


def choose_components(data, cap=8):
    n_samples, n_features = data.shape
    return max(1, min(cap, n_samples - 1, n_features))


def pca_reconstruct(data, n_components):
    n_components = max(1, min(n_components, data.shape[0] - 1, data.shape[1]))
    pca = PCA(n_components=n_components, random_state=SEED)
    z = pca.fit_transform(data)
    recon = pca.inverse_transform(z)
    return recon, z, pca


def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def two_sample_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b), 80)
    a = a[:n]
    b = b[:n]
    aa = pairwise_distances(a, a).mean()
    bb = pairwise_distances(b, b).mean()
    ab = pairwise_distances(a, b).mean()
    return float(2.0 * ab - aa - bb)


def preview_array(ax, sample, shape, title):
    arr = np.asarray(sample)
    if int(np.prod(shape)) == arr.size and shape[0] > 1 and shape[1] > 1:
        ax.imshow(arr.reshape(shape), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.plot(arr.ravel(), marker="o")
    ax.set_title(title, fontsize=9)


def show_ladder_preview(ladder):
    fig, axes = plt.subplots(1, len(ladder), figsize=(14, 2.8))
    for ax, rung in zip(axes, ladder):
        preview_array(ax, rung["data"][0], rung["shape"], rung["name"])
    plt.tight_layout()
    return fig


## The concept, built once (D1)
A residual conditioning branch adds structure to frozen features: $h_{out}=h_{base}+r_{cond}(c)$. The lesson uses $h_{base}=(0.4,0.1)$ and $r_{cond}=(0.3,-0.2)$, so $h_{out}=(0.7000,-0.1000)$ and $\lVert rVert_2=0.3606$.

In [ ]:

def residual_conditioned_generator(h_base, residual):
    h_base = np.asarray(h_base, dtype=float)
    residual = np.asarray(residual, dtype=float)
    h_out = h_base + residual
    residual_norm = float(np.linalg.norm(residual))
    return h_out, residual_norm

h_base = np.array([0.4, 0.1])
r_cond = np.array([0.3, -0.2])
h_out, residual_norm = residual_conditioned_generator(h_base, r_cond)
assert np.allclose(h_out, np.array([0.7, -0.1]))
assert round(residual_norm, 4) == 0.3606
print("h_out", np.round(h_out, 4), "residual_norm", round(residual_norm, 4))


For the ladder, a frozen low-rank PCA reconstruction plays the base generator. A condition vector predicts a residual correction; zero-like strength starts safe and then grows, matching the ControlNet idea of steering without immediately corrupting the base model.

In [ ]:

def condition_features(data):
    x = np.asarray(data, dtype=float)
    centered = x - x.mean(axis=1, keepdims=True)
    magnitude = np.abs(centered)
    coarse = np.stack([x.mean(axis=1), x.std(axis=1), magnitude.mean(axis=1)], axis=1)
    return coarse


def conditioned_residual_fit(data, strength=0.75, base_components=2):
    base_components = min(base_components, data.shape[1], data.shape[0] - 1)
    base, z, pca = pca_reconstruct(data, n_components=base_components)
    residual_target = data - base
    cond = condition_features(data)
    cond_aug = np.c_[np.ones(len(cond)), cond]
    weights = np.linalg.pinv(cond_aug) @ residual_target
    residual = strength * (cond_aug @ weights)
    generated = base + residual
    control_error = mse(data, generated)
    return generated, control_error, base, residual


## The dataset ladder (D1–D5)
We use the same F9 ladder inline for every topic: a one-dimensional Gaussian, two moons, a mixture, tiny sklearn digits, and a face rung. The face rung uses a local Olivetti cache when present and otherwise a deterministic no-download synthetic face fallback.

In [ ]:

ladder = make_f9_ladder()
for index, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(f"D{index}: {rung['name']} | shape={data.shape} | sample_shape={rung['shape']} | kind={rung['kind']}")
show_ladder_preview(ladder)


## Run the SAME method across D1-D5
The same reusable method is applied to every rung and one plan metric is collected per rung.

In [ ]:

results = []
for level, rung in enumerate(ladder, start=1):
    data = rung["data"]
    generated, metric, base, residual = conditioned_residual_fit(data)
    results.append({"level": level, "name": rung["name"], "metric": metric, "sample": generated[0], "shape": rung["shape"]})
    print(f"D{level} {rung['name']}: reconstruction/control error={metric:.5f}, residual_norm={np.linalg.norm(residual[0]):.4f}")


## Results visualization
The closing figure has generated-sample panels for D1-D5 plus the requested metric curve.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(14, 2.8))
for ax, result in zip(axes, results):
    preview_array(ax, result["sample"], result["shape"], result["name"])
plt.suptitle("Generated or reconstructed samples by rung")
plt.tight_layout()

plt.figure(figsize=(6, 3.2))
plt.plot([r["level"] for r in results], [r["metric"] for r in results], marker="o")
plt.xticks([r["level"] for r in results], [r["name"].split()[0] for r in results])
plt.ylabel("reconstruction / control error")
plt.xlabel("F9 rung")
plt.title("Residual conditioned generator: metric vs. complexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## Pitfall on D5: late condition or non-zero disruption
If conditioning arrives too late, the spatial structure has already been discarded. If the branch starts too strong, it disrupts the frozen generator before learning useful corrections; we compare a global late condition with an early residual fit.

In [ ]:

d5 = ladder[-1]
data = d5["data"]
late_base, _, _ = pca_reconstruct(data, n_components=2)
late_condition = np.repeat(data.mean(axis=1, keepdims=True), data.shape[1], axis=1)
late_generated = 0.65 * late_base + 0.35 * late_condition
late_error = mse(data, late_generated)
early_generated, early_error, early_base, early_residual = conditioned_residual_fit(data, strength=0.75, base_components=4)
print(f"late/uncontrolled error={late_error:.5f}")
print(f"early residual error={early_error:.5f}")
fig, axes = plt.subplots(1, 3, figsize=(7, 2.4))
preview_array(axes[0], data[0], d5["shape"], "target D5")
preview_array(axes[1], late_generated[0], d5["shape"], "late condition")
preview_array(axes[2], early_generated[0], d5["shape"], "early residual")
plt.tight_layout()


## Evaluate it + Practice
- Track `reconstruction/control error` against a no-skill baseline such as random samples, unconditioned reconstruction, or independent frames.
- Run a cheap sanity check: dimensions match, finite metrics, and generated samples stay in the training range.
- Ablation: inject the condition only as a late global scalar or initialize the residual branch too large; the metric should get worse.
- Failure signals: D5 looks plausible in one panel but the metric curve jumps, indicating artifacts hidden by small previews.

Practice 1: change the latent or conditioning dimension and predict which rung changes most.


Practice 2: replace the condition features with edge-like finite differences and compare D4 versus D5.

Practice 3: sweep the residual strength from zero-like to disruptive and plot the error curve.